# 08 Q2 runs (preregistration section 9)

Swap circle groups on an operand token and measure the Q1 supernodes at "=" (`helixadd.q2`).

1. **Pilot** (9.10, declared, not counted): 50 prompts, the swaps, the ceiling and 5 random sets per kind and operand. Saved to `results/q2_pilot/`. Checks the code, the timing, and that the ceiling moves the supernodes at all.
2. **Main run**, only after the freeze: all 998 prompts, 100 random sets. Saved to `results/q2_runs/`, resumable. Run as Save Version, Save & Run All.

Needs `results/number_acts/data.npz` in the repo. Setup: GPU T4, Internet on, secret `HF_TOKEN` ticked.

In [ ]:
!rm -rf helix_addition && git clone https://github.com/amark-23/helix_addition.git
!pip install -q git+https://github.com/safety-research/circuit-tracer@v0.5.3

In [ ]:
import os, sys
os.chdir("/kaggle/working/helix_addition")
sys.path.insert(0, "/kaggle/working/helix_addition/src")
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(UserSecretsClient().get_secret("HF_TOKEN"))

In [ ]:
import numpy as np, time
from helixadd import model as hmodel, q1, q2
model = hmodel.load_model()
data = dict(np.load("results/number_acts/data.npz"))
pairs = q1.eval_pairs()
print(len(pairs), "evaluation prompts")

## 1. Pilot

In [ ]:
RUN_MAIN = False   # set to True only after the Q2 freeze
t = time.time()
q2.run_all(model, data, pairs[-50:], folder="results/q2_pilot", n_random=5)
print(f"{time.time() - t:.0f} s for the pilot")
q2.summary("results/q2_pilot").round(3)

## 2. Main run (after the freeze)

In [ ]:
if RUN_MAIN:
    q2.run_all(model, data, pairs)
    print(q2.summary().round(3))

In [ ]:
from IPython.display import FileLink
os.chdir("/kaggle/working")
!zip -qr q2_results.zip helix_addition/results/q2_pilot helix_addition/results/q2_runs
FileLink("q2_results.zip")